# Sequence Models and the Emergence of the Transformer

According to *Generative AI Foundations in Python*, modeling long-range dependencies required more sophisticated network architectures, leading to the use of RNNs. Recurrent Neural Networks process data sequences by iterating through each element while maintaining a dynamic internal state. This was further improved by Long Short-Term Memory (LSTM) networks, which applied a unique gating architecture to control the flow of information, maintaining and accessing information over long sequences without suffering from the vanishing gradient problem.

Concurrently, Convolutional Neural Networks (CNNs) were adapted for NLP to extract hierarchical features using convolutional layers over local n-gram windows. However, the true paradigm shift occurred in 2017 with the introduction of the Transformer architecture by Vaswani et al. The Transformer applied a self-attention mechanism, allowing each element in the input sequence to focus on distinct parts of the sequence, capturing dependencies regardless of their positions. This sequence-to-sequence learning model became the foundation for all modern generative language models.


In [ ]:
#| echo: false
#| eval: true
import seaborn as sns
import matplotlib.pyplot as plt

def set_economist_theme():
    sns.set_theme(
        context="talk",
        style="whitegrid",
        rc={
            "font.family": "serif",
            "axes.titlesize": 14,
            "axes.titleweight": "bold",
            "axes.labelsize": 12,
            "axes.labelweight": "bold",
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "grid.color": "#d9d9d9",
            "grid.linewidth": 0.6,
            "axes.edgecolor": "#333333",
            "axes.linewidth": 0.8,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "lines.linewidth": 1.0,
            "figure.figsize": (8, 4),
            "figure.dpi": 120,
            "legend.frameon": False,
        }
    )
set_economist_theme()


# A Brief History of Language Models {#sec-llm-history}

## From Text to Intelligence: The Language Modeling Imperative

Language is a prominent ability in human beings to express and communicate, which develops
in early childhood and evolves over a lifetime. Machines, however, cannot naturally grasp
the abilities of understanding and communicating in the form of human language, unless
equipped with powerful artificial intelligence (AI) algorithms. It has been a longstanding
research challenge to enable machines to read, write, and communicate like humans
[@turing-test].

Technically, **language modeling (LM)** is one of the major approaches to advancing
language intelligence of machines. In general, LM aims to model the generative likelihood
of word sequences, so as to predict the probabilities of future (or missing) tokens.
The research of LM has received extensive attention in the literature, which can be divided
into four major development stages [@zhao2023survey]:

- **Statistical language models (SLM)**. SLMs are developed based on **statistical
  learning** methods that rose in the 1990s. The basic idea is to build the word prediction
  model based on the Markov assumption — predicting the next word based on the most recent
  context. The SLMs with a fixed context length $n$ are also called $n$-gram language
  models (bigram, trigram). SLMs suffer from the *curse of dimensionality*: estimating
  high-order transition probabilities requires exponential sample size, so smoothing
  strategies (back-off, Good–Turing) were needed to alleviate data sparsity.

- **Neural language models (NLM)**. NLMs characterize the probability of word sequences
  by neural networks — MLP and RNNs. The seminal work of @Bengio-JMLR-2003-A introduced
  the concept of **distributed representation** of words and built word prediction
  conditioned on aggregated context features. word2vec [@Mikolov-NIPS-2013] further
  simplified this to a shallow network and demonstrated effective representations across
  NLP tasks. These studies initiated the use of language models for *representation
  learning*, beyond simple sequence modeling.

- **Pre-trained language models (PLM)**. ELMo [@Peters-NAACL-2018] captured
  context-aware word representations by pre-training a bidirectional LSTM and then
  fine-tuning for downstream tasks. BERT [@Devlin-NAACL-2019-BERT] extended this with the
  Transformer architecture and masked pre-training, establishing the **pre-training and
  fine-tuning** learning paradigm. A large number of follow-up models (GPT-2, BART,
  RoBERTa) were built on this foundation.

- **Large language models (LLM)**. Scaling PLMs in model size or data volume leads to
  improved capacity following the **scaling law** [@Kaplan-arxiv-2020-Scaling]. Large-sized
  PLMs (GPT-3 at 175B parameters; PaLM at 540B) display **emergent abilities** not seen in
  smaller models — most notably, **in-context learning** (solving new tasks from a handful
  of examples without gradient updates). These models are now general-purpose task solvers
  rather than task-specific tools.

## The Pivotal Shift: From Task-Specific to General-Purpose

From the perspective of task solving, the four generations of language models have
exhibited different levels of model capacity:

| Generation | Representative Models | Task Approach |
|---|---|---|
| SLM | n-gram, SRILM | Enhance task-specific pipelines (IR, ASR) |
| NLM | word2vec, fastText, ELMo | Learn task-agnostic representations |
| PLM | BERT, GPT-2, BART | Pre-train once, fine-tune per task |
| LLM | GPT-3, GPT-4, PaLM, LLaMA | General-purpose solver via prompting |

The key conceptual jump is from **language modeling** to **task solving** — and from
*training-time* adaptation (fine-tuning) to *inference-time* steering (prompting).
This is the paradigm shift we trace through Modules 3 and 4.


# Introduction: The Challenge of Sequential Data {#sec-sequential}

Language is inherently sequential. A sentence is not a bag of independent tokens — the
meaning of a word depends on what came before it (and sometimes after it). This simple
observation has deep consequences for model design.

When we moved from Modules 1–2 (bag-of-words, TF-IDF, static embeddings) to sequences,
we faced a new challenge: **how do we encode order and long-range dependencies?**

The approaches fall into a clear historical arc:

1. **N-gram models** — Markov assumption: only the last $n-1$ tokens matter.
2. **RNNs** — maintain a hidden state that summarizes all previous tokens.
3. **Attention mechanisms** — directly weight all prior tokens without a bottleneck.
4. **Transformers** — replace recurrence entirely with multi-head self-attention.

The rest of this lecture builds this arc from the ground up.

{{< include ./m03_l01_content/sequence.qmd >}}

{{< include ./m03_l01_content/text-sequence.qmd >}}

{{< include ./m03_l01_content/index.qmd >}}


# From Recurrence to Attention {#sec-recurrence-to-attention}

## The RNN Bottleneck

RNNs process sequences token by token, updating a hidden state $h_t$ at each step:

$$
\begin{align}
h_t = f(x_t,\, h_{t-1})
\end{align}
$$

This design has two fundamental weaknesses:

1. **Sequential computation** — each step depends on the previous, so training cannot be
   parallelized across time steps. Long sequences become slow.
2. **Information bottleneck** — the entire history must be compressed into a fixed-size
   vector $h_t$. For long sequences, early tokens are increasingly hard to recall. This
   is the **vanishing gradient** problem.

LSTM and GRU cells mitigate (but do not eliminate) the vanishing gradient by adding gated
memory cells. But the bottleneck remains.

## The Attention Insight

What if, instead of compressing history into a single vector, we allowed the model to
**directly access any past token** when processing the current one?

This is the attention mechanism. Given a **query** $q$ and a set of **key–value** pairs
$(k_i, v_i)$, attention computes a weighted sum of values:

$$
\begin{align}
\text{Attention}(q, K, V) = \sum_i \alpha_i \, v_i, \qquad
\alpha_i = \text{softmax}\!\left(\frac{q \cdot k_i}{\sqrt{d_k}}\right)
\end{align}
$$

The weights $\alpha_i$ measure how relevant each key $k_i$ is to the current query $q$.
The division by $\sqrt{d_k}$ prevents the dot products from growing too large in high
dimensions (scaled dot-product attention, @vaswani2017attention).

## Multi-Head Attention

A single attention function may attend to one type of relationship. **Multi-head attention**
runs $H$ attention heads in parallel, each projecting queries, keys, and values into a
different subspace:

$$
\begin{align}
\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H)\,W^O \\
\text{where} \quad \text{head}_i = \text{Attention}(Q W_i^Q,\, K W_i^K,\, V W_i^V)
\end{align}
$$

Different heads can learn to attend to syntactic structure, semantic similarity, coreference,
and long-range dependencies simultaneously.

## Why Attention Replaces Recurrence

| Property | RNN | Transformer (Self-Attention) |
|---|---|---|
| Long-range dependency | ✗ (degrades) | ✓ (direct path) |
| Parallel training | ✗ (sequential) | ✓ (all positions at once) |
| Constant path length | ✗ ($O(n)$) | ✓ ($O(1)$) |
| Memory per token | Fixed ($d_h$) | $O(n)$ (attention matrix) |

The trade-off is memory: transformers store an $n \times n$ attention matrix per layer,
which is expensive for very long sequences. Modern variants (FlashAttention, sparse
attention) address this.


# Transformers and the Attention Mechanism {#sec-transformers}

## The Transformer Timeline

The transformer architecture [@vaswani2017attention] introduced in 2017 ("Attention is
All You Need") revolutionized NLP and laid the groundwork for modern LLMs.

![Timeline of transformer-based models](./M03_lecture01_figures/02-13.png){width=80% fig-align=center fig-alt="Timeline of transformer-based models" #fig-timeline}

## Transformer Architecture

![Transformer Architecture](./M03_lecture01_figures/transformer-archicture.jpg){width=80% fig-align=center fig-alt="Transformer architecture diagram" #fig-transformer-architecture}

The full Transformer stack (encoder + decoder) adds several key components beyond
multi-head attention:

- **Positional encoding** — since self-attention is order-agnostic, sinusoidal or learned
  position embeddings are added to the token embeddings.
- **Feed-forward sublayer** — a two-layer MLP applied position-wise after each attention
  block.
- **Layer normalization + residual connections** — stabilize training of deep stacks.
- **Masking** — encoder uses full bidirectional attention; decoder uses causal masking to
  prevent attending to future tokens.


## What's Next: Token Representations and Semantic Geometry

In Lecture 2 (M03_LN2) we move from *architecture* to *representation*: how do
transformers convert tokens into dense vectors that capture semantic meaning?

We will cover:

- Subword tokenization (BPE, WordPiece, SentencePiece) — how raw text becomes token IDs
- Static vs. contextual embeddings — word2vec vs. BERT-style representations
- Semantic geometry — cosine similarity, analogy arithmetic, clustering in embedding space
- Self-attention code demonstrations — computing attention weights from scratch


### References {.unnumbered}

::: {#refs}
:::
